### Music Discovery — Exploração e seleção de músicas

**Objetivo:**

Explorar a base de músicas do Spotify e permitir a seleção manual de faixas para uso no projeto. A busca pode ser feita pelo nome da música, artista ou gênero, com a possibilidade de montar uma lista de até 10 músicas.

**Etapas**

- **Preparação da base:** aplicação da função `preprocessing` desenvolvida para o projeto.
- **Inspeção dos dados:** verificação dos metadados e das features utilizadas.
- **Seleção de músicas:** interface interativa para buscar, adicionar e remover músicas da lista.

In [12]:
import sys
import pandas as pd
import ipywidgets as widgets

# Adiciona a pasta raiz do projeto ao caminho de busca (o preprocessing.py fica fora da pasta notebooks)
sys.path.append("..")

from app.preprocessing.preprocessing import preprocessing

from IPython.display import display, clear_output

### Spotify Tracks Dataset

In [13]:
# Carrega dados de músicas do Spotify
df = pd.read_csv("../raw_data/spotify-tracks-dataset-detailed.csv")
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [14]:
# Verifica quantidade de linhas da base
len(df)

114000

### 1. Preparação dos dados

A função `preprocessing` reúne as etapas de preparação da base em um único lugar, permitindo reaplicar o mesmo tratamento utilizado nas demais análises do projeto.

Nesta etapa, a função retorna:
- `X_scaled`: matriz com as features escaladas;
- `df_metadata`: metadados das músicas;
- `scaler`: objeto utilizado no escalonamento;
- `features`: lista das features utilizadas.

#### 1.1 Aplicação do Pre Processing

In [16]:
# Aplica o pre processing na base
X_scaled, df_metadata, scaler, features = preprocessing(df)

In [17]:
# Verifica o formato da matriz de features escaladas
X_scaled.shape

(80175, 8)

In [18]:
# Verifica os metadados das músicas
df_metadata.head()

,track_id,artists,track_name,genres
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]"
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]"
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic]
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic]
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic]


In [19]:
# Cria o dataframe com as features escaladas
df_scaled = pd.DataFrame(X_scaled, columns=features)
df_scaled.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,0.676937,-0.701560,0.323475,0.776805,-0.854487,-0.551196,0.960811,-1.182342
1,-0.815682,-1.857298,-1.792992,-0.041318,1.801679,-0.551182,-0.762295,-1.535282
2,-0.710732,-1.101171,-0.279443,-0.293991,-0.324922,-0.551199,-1.327690,-1.574441
3,-1.713586,-2.274147,-2.051270,-0.531946,1.745089,-0.550983,-1.239227,1.993142
4,0.338766,-0.772080,-0.268748,-0.332015,0.446492,-0.551199,-1.146917,-0.098203


In [20]:
# Une os metadados com as features escaladas
df_final = pd.concat([df_metadata, df_scaled], axis=1)
df_final.head()

,track_id,artists,track_name,genres,danceability,energy,loudness,speechiness,acousticness,instrumentalness,valence,tempo
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,"[acoustic, j-pop, singer-songwriter, songwriter]",0.676937,-0.701560,0.323475,0.776805,-0.854487,-0.551196,0.960811,-1.182342
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,"[acoustic, chill]",-0.815682,-1.857298,-1.792992,-0.041318,1.801679,-0.551182,-0.762295,-1.535282
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,[acoustic],-0.710732,-1.101171,-0.279443,-0.293991,-0.324922,-0.551199,-1.327690,-1.574441
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,[acoustic],-1.713586,-2.274147,-2.051270,-0.531946,1.745089,-0.550983,-1.239227,1.993142
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,[acoustic],0.338766,-0.772080,-0.268748,-0.332015,0.446492,-0.551199,-1.146917,-0.098203


### 2. Exploração e seleção de músicas

Com os dados preparados, a próxima etapa é disponibilizar uma busca para explorar o catálogo e selecionar músicas manualmente.

A interface permite:
- buscar por todos os campos ou filtrar por música, artista e gênero;
- visualizar sugestões conforme o termo digitado;
- adicionar até 10 músicas à seleção;
- remover uma música específica ou limpar a lista.

In [21]:

base = df_final
MAX_MUSICAS = 10

# --- 1. Colunas de busca (feito uma vez só) ---
def lista_para_texto(g):
    if isinstance(g, (list, tuple, set)):
        return " ".join(map(str, g))
    return "" if pd.isna(g) else str(g)

col_genero = next((c for c in base.columns if c.strip().lower() in ("genres", "genre", "track_genre")), None)

base["_musica"] = base["track_name"].fillna("").astype(str).str.lower()
base["_artista"] = base["artists"].fillna("").astype(str).str.lower()
base["_genero"] = base[col_genero].apply(lista_para_texto).str.lower() if col_genero else ""
base["_tudo"] = base["_musica"] + " " + base["_artista"] + " " + base["_genero"]

CAMPOS = {"Tudo": "_tudo", "Música": "_musica", "Artista": "_artista"}
if col_genero:
    CAMPOS["Gênero"] = "_genero"

# --- 2. Estado e widgets ---
selecionadas = []      # índices do df_final das músicas escolhidas
atualizando = {"on": False}   # evita disparar o clique quando a lista é redesenhada

campo = widgets.Dropdown(options=list(CAMPOS), value="Tudo", description="Campo:")
caixa = widgets.Text(
    placeholder="Digite uma música, artista ou gênero...",
    description="Buscar:",
    layout=widgets.Layout(width="500px"),
)
lista = widgets.Select(options=[], rows=8, layout=widgets.Layout(width="500px"))

contador = widgets.HTML()
escolhidas = widgets.Select(options=[], rows=10, layout=widgets.Layout(width="500px"))
btn_remover = widgets.Button(description="Remover", icon="trash")
btn_limpar = widgets.Button(description="Limpar tudo")

def rotulo(idx):
    return f"{base.at[idx, 'track_name']} — {base.at[idx, 'artists']}"

def redesenhar_escolhidas(aviso=""):
    escolhidas.options = [(rotulo(i), i) for i in selecionadas]
    n = len(selecionadas)
    status = " ✅ pronto!" if n == MAX_MUSICAS else ""
    contador.value = f"<b>Selecionadas: {n}/{MAX_MUSICAS}</b>{status} {aviso}"

# --- 3. Autocompletar ---
def atualizar_sugestoes(change=None):
    termo = caixa.value.strip().lower()
    atualizando["on"] = True
    if len(termo) < 2:
        lista.options = []
    else:
        mask = base[CAMPOS[campo.value]].str.contains(termo, regex=False)
        resultados = base[mask].head(10)
        lista.options = [
            (("✓ " if idx in selecionadas else "") + rotulo(idx), idx)
            for idx in resultados.index
        ]
        lista.index = None
    atualizando["on"] = False

def adicionar(change):
    idx = change["new"]
    if atualizando["on"] or idx is None:
        return
    ids = set(base.loc[selecionadas, "track_id"]) if selecionadas else set()
    aviso = ""
    if len(selecionadas) >= MAX_MUSICAS:
        aviso = "⚠️ limite atingido, remova uma para adicionar outra."
    elif base.at[idx, "track_id"] in ids:
        aviso = "⚠️ essa música já está na lista."
    else:
        selecionadas.append(idx)
    atualizando["on"] = True
    lista.index = None          # permite clicar na mesma sugestão de novo
    atualizando["on"] = False
    redesenhar_escolhidas(aviso)
    atualizar_sugestoes()       # atualiza os ✓ nas sugestões

def remover(_):
    if escolhidas.value in selecionadas:
        selecionadas.remove(escolhidas.value)
    redesenhar_escolhidas()
    atualizar_sugestoes()

def limpar(_):
    selecionadas.clear()
    redesenhar_escolhidas()
    atualizar_sugestoes()

caixa.observe(atualizar_sugestoes, names="value")
campo.observe(atualizar_sugestoes, names="value")
lista.observe(adicionar, names="value")
btn_remover.on_click(remover)
btn_limpar.on_click(limpar)

redesenhar_escolhidas()
display(campo, caixa, lista, contador, escolhidas, widgets.HBox([btn_remover, btn_limpar]))

Dropdown(description='Campo:', options=('Tudo', 'Música', 'Artista', 'Gênero'), value='Tudo')

Text(value='', description='Buscar:', layout=Layout(width='500px'), placeholder='Digite uma música, artista ou…

Select(layout=Layout(width='500px'), options=(), rows=8, value=None)

HTML(value='<b>Selecionadas: 0/10</b> ')

Select(layout=Layout(width='500px'), options=(), rows=10, value=None)

### 3. Visualização da seleção

Após escolher as músicas na interface, execute a célula abaixo para visualizar a seleção em um DataFrame.

In [23]:
# Cria um dataframe com as músicas selecionadas
df_selecionadas = df_final.loc[
    selecionadas,
    ["track_id", "artists", "track_name", "genres"]
].reset_index(drop=True)

# Verifica as músicas selecionadas
df_selecionadas

,track_id,artists,track_name,genres
0,0nLiqZ6A27jJri2VCalIUs,Metallica,Nothing Else Matters,"[hard-rock, metal, rock]"
1,0hCB0YR03f6AmQaHbwWDe8,Led Zeppelin,Whole Lotta Love - 1990 Remaster,[hard-rock]
2,6GyDY0yE47rfk8pcuKhioh,Queens of the Stone Age,Make It Wit Chu,[blues]
3,32BEasyNQgqXQJKoVa0fUw,Greta Van Fleet,Black Smoke Rising,"[blues, hard-rock]"
4,5fpq1wF8xa5tSSlcKHdmGQ,Bring Me The Horizon,sTraNgeRs,"[metal, metalcore]"
5,0QRxtcxL31dRAeiUUuENPu,Sepultura,Territory,"[death-metal, groove]"
6,3gVhsZtseYtY1fMuyYq06F,Rammstein,Sonne,"[alt-rock, german, industrial, metal]"
7,1M8LMTTEajUuw7AN0g4f4i,Megadeth,Holy Wars...The Punishment Due - Remastered,"[alt-rock, alternative, hard-rock]"
8,3TO7bbrUKrOSPGRTB5MeCz,Pink Floyd,Time,[psych-rock]
9,0upLyFR8Rr52ZpMp5esQoq,Van Halen,You Really Got Me - 2015 Remaster,[hard-rock]
